# 22 — Package the advanced track with Podman
The Compose file starts PostgreSQL, Redis, the API, and Streamlit. Ollama
runs on the host; Langfuse is an explicitly configured separate endpoint.



**Advanced follow-up to lesson 13 (project).**


## Problem

A multi-service demo needs reproducible local startup.


## Concept

Podman Compose packages API, UI, PostgreSQL, and Redis while Ollama stays local.

A service manifest defines dependencies, health checks, private storage, and localhost-only ports.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out   ◀ YOU ARE HERE
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
host Ollama → API → PostgreSQL/Redis; Streamlit → API; Podman Compose owns services
```

Containerfile → API/UI; Compose → PostgreSQL/Redis/volumes. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Check Podman availability and service files

Check Podman availability and service files.


In [ ]:
import shutil
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.common import ROOT

print("Podman installed:", bool(shutil.which("podman")))
print("Compose file exists:", (ROOT / "compose.yaml").is_file())
print("Containerfile exists:", (ROOT / "Containerfile").is_file())



## Start services after setting credentials in .env
Run `podman compose up --build` from the repository root. Verify /ready and
the UI at localhost:8501. `podman compose` needs a Compose provider installed.



## Step 2 — Read readiness URLs and start Compose separately

Read readiness URLs and start Compose separately.


In [ ]:
print("API readiness URL: http://localhost:8000/ready")
print("Streamlit URL: http://localhost:8501")


## Failure mode to know about

An image build does not prove Ollama is reachable from the API container. Check both model URLs.


## Production implication

Pin image digests, manage secrets, and rehearse restart/recovery before non-demo deployment.


## Your turn

Check service readiness and inspect which data survives a restart.

**Checkpoint:** The stack starts with explicit credentials and healthy dependencies.
